# SAE Two-Concept Steering with NDCG

This is the two-concept version of the original steering notebook.

It keeps the same **item-based neuron discovery**:

1. take item embeddings,
2. pass them through the SAE,
3. group item SAE activations by metadata concept,
4. select the strongest neuron for each concept.

Then, during recommendation, it modifies the **user SAE representation** so that one concept is amplified while another concept is suppressed.

The default example below is:

- amplify **Horror**
- suppress **Comedy**

For evaluation it reports four conditions:

- Baseline
- Amplify only
- Suppress only
- Amplify + Suppress simultaneously

It also reports Concept Fraction@K, Concept Hit@K, NDCG@K, ΔNDCG, and NDCG retention.


In [1]:
from recbole.quick_start import load_data_and_model, run_recbole
import torch
import pandas as pd
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import NeuMF
from recbole.trainer import Trainer
from recbole.utils import get_model, get_trainer, init_seed, init_logger
from collections import defaultdict
import os
from recbole.quick_start import load_data_and_model
import rbo as rbo_lib
import rbo


In [2]:
import yaml
from recbole.quick_start import load_data_and_model

#with open('YAML_files/CBM_config.yaml', 'r') as f:
 #   config_overrides = yaml.safe_load(f)

#config_overrides['train_neg_sample_args']={'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}

config, SAE_model, dataset, train_data, valid_data, test_data = load_data_and_model(
        config_dict={'data_path': 'dataset/ml-1m', 'dataset': 'ml-1m', 'base_path':'./saved/sasrec_ml-1m.pth'}, 
        model_file="saved/SASRec_Mon_SAE-Aug-31-2026_11-05-06.pth")


SAE_model.eval()
device = config['device']

/home/mvarasteh/post-hoc/recbole/quick_start/quick_start.py:249: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_file, map_location=torch.device(


load_col = {'inter': ['user_id', 'item_id', 'timestamp']}
unload_col = None
unused_col = None
additional_feat_suffix = None
rm_dup_inter = None
val_interval = None
filter_inter_by_user_or_item = True
user_inter_num_interval = [0,inf)
item_inter_num_interval = [0,inf)
alias_of_user_id = None
alias_of_item_id = None
alias_of_entity_id = None
alias_of_relation_id = None
preload_weight = None
normalize_field = None
normalize_all = None
ITEM_LIST_LENGTH_FIELD = item_length
LIST_SUFFIX = _list
MAX_ITEM_LIST_LENGTH = 50
POSITION_FIELD = position_id
HEAD_ENTITY_ID_FIELD = head_id
TAIL_ENTITY_ID_FIELD = tail_id
RELATION_ID_FIELD = relation_id
ENTITY_ID_FIELD = entity_id
benchmark_filename = None

Other Hyper Parameters: 
worker = 0
wandb_project = SASRec_Mon-project
shuffle = True
require_pow = False
enable_amp = False
enable_scaler = False
transform = None
nproc = 1
numerical_features = []
discretization = None
kg_reverse_r = False
entity_kg_num_interval = [0,inf)
relation_kg_num_interval = [

In [3]:
saved_concepts=pd.read_pickle("/home/mvarasteh/post-hoc/dataset/ml-1m/saved_concept_individual_items.pkl")
concept_names=saved_concepts['concept_names']
item_concepts_np=saved_concepts['item_concepts']

In [4]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm


## NDCG@K helper

This implementation uses RecBole's `positive_u` and `positive_i` mappings and therefore works for full-sort evaluation batches even when a batch contains multiple positive items.


In [5]:
def compute_ndcg_from_topk(
    topk,
    positive_u,
    positive_i,
    k,
):
    """
    Compute user-level NDCG@K from a RecBole full-sort evaluation batch.

    Parameters
    ----------
    topk : torch.Tensor
        Shape [B, K]. Recommended item IDs for each user row.

    positive_u : torch.Tensor
        Row indices identifying which user each positive item belongs to.

    positive_i : torch.Tensor
        Positive test-item IDs corresponding to positive_u.

    k : int
        Ranking cutoff.

    Returns
    -------
    ndcg : torch.Tensor
        NDCG@K for each user row.

    valid_mask : torch.Tensor
        True for user rows with at least one positive test item.
    """

    device = topk.device
    batch_size = topk.size(0)

    positive_u = positive_u.to(
        device=device,
        dtype=torch.long,
    )

    positive_i = positive_i.to(
        device=device,
        dtype=torch.long,
    )

    # Binary relevance matrix: relevance[user, rank]
    relevance = torch.zeros(
        (batch_size, k),
        dtype=torch.float32,
        device=device,
    )

    if positive_i.numel() > 0:
        # Compare each positive item against the corresponding user's Top-K.
        matches = (
            topk[positive_u]
            == positive_i.unsqueeze(1)
        )

        positive_pair_idx, rank_idx = torch.where(matches)

        if positive_pair_idx.numel() > 0:
            user_rows = positive_u[positive_pair_idx]

            relevance[
                user_rows,
                rank_idx
            ] = 1.0

    # DCG discounts
    ranks = torch.arange(
        1,
        k + 1,
        device=device,
        dtype=torch.float32,
    )

    discounts = 1.0 / torch.log2(
        ranks + 1.0
    )

    dcg = (
        relevance * discounts
    ).sum(dim=1)

    # Number of positive items for each user row
    num_positives = torch.bincount(
        positive_u,
        minlength=batch_size,
    )

    valid_mask = num_positives > 0

    # Ideal DCG
    cumulative_discounts = torch.cumsum(
        discounts,
        dim=0,
    )

    ideal_length = torch.clamp(
        num_positives,
        min=0,
        max=k,
    )

    idcg = torch.zeros(
        batch_size,
        dtype=torch.float32,
        device=device,
    )

    nonzero = ideal_length > 0

    idcg[nonzero] = (
        cumulative_discounts[
            ideal_length[nonzero] - 1
        ]
    )

    ndcg = torch.zeros_like(dcg)

    ndcg[nonzero] = (
        dcg[nonzero]
        / idcg[nonzero]
    )

    return ndcg, valid_mask


## 1. Discover concept neurons from ITEMS

This section deliberately uses **item embeddings**, not user profiles, to assign semantic concepts to SAE neurons.


In [6]:
SAE_model.eval()

with torch.no_grad():

    item_embeddings = (
        SAE_model
        .item_embedding
        .weight
        .detach()
    )

    item_activations = (
        SAE_model
        .sae
        .encode(item_embeddings)
        .cpu()
        .numpy()
    )

print(
    "Item activations shape:",
    item_activations.shape
)

Item activations shape: (3417, 142)


## Top1 neuron for each cocnept

In [7]:
concept_mean_activations = {}

for concept_idx, concept_name in enumerate(
    concept_names
):

    # Find items belonging to this concept
    item_ids = np.where(
        item_concepts_np[:, concept_idx] > 0
    )[0]

    # Remove RecBole padding item
    item_ids = item_ids[
        item_ids != 0
    ]

    if len(item_ids) == 0:
        continue

    # SAE activations of items in this concept
    activations = item_activations[
        item_ids
    ]

    # Mean activation of every SAE neuron
    mean_activation = (
        activations.mean(axis=0)
    )

    concept_mean_activations[
        concept_name
    ] = mean_activation


# Convert to DataFrame
concept_activation_df = pd.DataFrame(
    concept_mean_activations,
    index=[
        f"Neuron_{i}"
        for i in range(
            item_activations.shape[1]
        )
    ]
).T

best_neuron_by_concept = {}

for concept_name in (
    concept_activation_df.index
):

    best_neuron_name = (
        concept_activation_df
        .loc[concept_name]
        .idxmax()
    )

    # Example:
    # "Neuron_5" -> 5
    neuron_idx = int(
        best_neuron_name.split("_")[1]
    )

    best_neuron_by_concept[
        concept_name
    ] = neuron_idx




In [8]:
TOP_K = 3

topk_neurons_with_values = {}

for concept_name in concept_activation_df.index:

    topk = (
        concept_activation_df
        .loc[concept_name]
        .sort_values(ascending=False)
        .head(TOP_K)
    )

    topk_neurons_with_values[
        concept_name
    ] = [
        (
            int(neuron_name.split("_")[1]),
            float(value)
        )
        for neuron_name, value in topk.items()
    ]
    

## 3. Two-concept full-sort prediction

The model source currently supports one built-in steering neuron. This notebook therefore performs the two-neuron intervention explicitly in the SAE latent representation.

Concept discovery is still item-based. The intervention itself is applied to the user latent vector because that is what changes the user's recommendation direction.


In [55]:
# Reconstruct the item side once.
# This is the same shared SAE used for item embeddings in the model.

with torch.no_grad():

    all_item_embeddings = (
        SAE_model
        .item_embedding
        .weight
        .detach()
    )

    all_item_latents = (
        SAE_model
        .sae
        .encode(all_item_embeddings)
    )

    reconstructed_item_embeddings = (
        SAE_model
        .sae
        .decode(all_item_latents)
        .clone()
    )

    # Match the model's _reconstruct_all_items():
    # item 0 is RecBole padding.
    reconstructed_item_embeddings[0] = 0.0


@torch.no_grad()
def two_concept_full_sort_predict(
    model,
    interaction,
    amplify_neuron,
    amplify_add,
    suppress_neuron,
    suppress_scale,
    reconstructed_item_embeddings,
):
    """
    Full-sort prediction after simultaneously:

      amplified neuron:
          z[:, amplify_neuron] += amplify_add

      suppressed neuron:
          z[:, suppress_neuron] *= suppress_scale

    Only the USER SAE representation is intervened on.
    Concept-neuron discovery was performed separately from ITEM activations.
    """

    item_seq = interaction[model.ITEM_SEQ]
    item_seq_len = interaction[model.ITEM_SEQ_LEN]

    # Frozen SASRec user/profile representation h.
    h = model._encode(
        item_seq,
        item_seq_len,
    )

    # Shared SAE latent representation of the user.
    z_user = (
        model
        .sae
        .encode(h)
        .clone()
    )

    # Amplify the first semantic direction.
    z_user[:, int(amplify_neuron)] *= float(
        amplify_add
    )

    # Suppress the second semantic direction.
    z_user[:, int(suppress_neuron)] *= float(
        suppress_scale
    )

    # SAE encoder outputs are non-negative.
    z_user = torch.clamp(
        z_user,
        min=0.0,
    )

    # Decode the modified user representation.
    h_hat = model.sae.decode(
        z_user
    )

    # Score against the unmodified reconstructed item table.
    scores = (
        h_hat
        @ reconstructed_item_embeddings.t()
    )

    return scores


## 4. Sanity check

With amplification `0` and suppression scale `1`, the external two-concept predictor should reproduce the model's ordinary unsteered SAE scores.


In [56]:
@torch.no_grad()
def check_two_concept_predictor(
    model,
    test_data,
    n_items,
    device,
    reconstructed_item_embeddings,
):
    model.eval()

    # Disable the model's built-in one-neuron steering.
    model.steer_neuron_idx = None
    model.steer_add = 0.0
    model.steer_scale = 1.0

    batch = next(iter(test_data))

    (
        interaction,
        history_index,
        positive_u,
        positive_i,
    ) = batch

    interaction = interaction.to(device)

    native_scores = (
        model
        .full_sort_predict(interaction)
        .view(-1, n_items)
    )

    external_scores = (
        two_concept_full_sort_predict(
            model=model,
            interaction=interaction,
            amplify_neuron=AMPLIFY_NEURON,
            amplify_add=0.0,
            suppress_neuron=SUPPRESS_NEURON,
            suppress_scale=1.0,
            reconstructed_item_embeddings=(
                reconstructed_item_embeddings
            ),
        )
        .view(-1, n_items)
    )

    max_diff = (
        native_scores
        - external_scores
    ).abs().max().item()

    print(
        "Maximum native-vs-external "
        "score difference:",
        max_diff
    )

    if max_diff <= 1e-4:
        print("Sanity check passed.")
    else:
        print(
            "WARNING: scores differ more than expected. "
            "Do not use the steering results until this is resolved."
        )

    return max_diff


predictor_max_diff = check_two_concept_predictor(
    model=SAE_model,
    test_data=test_data,
    n_items=test_data.dataset.item_num,
    device=device,
    reconstructed_item_embeddings=(
        reconstructed_item_embeddings
    ),
)


Maximum native-vs-external score difference: 3.9475560188293457


## 5. Define experimental conditions

The single-concept conditions are included as controls so you can distinguish the effect of amplification from the effect of suppression.

`AMPLIFY_ADD` is added to the amplified neuron's user activation.

`SUPPRESS_SCALE` multiplies the suppressed neuron's user activation:

- `1.0` = no suppression
- `0.5` = retain 50%
- `0.25` = retain 25%
- `0.0` = complete ablation


In [102]:
AMPLIFY_CONCEPT = "Drama"
SUPPRESS_CONCEPT = "Comedy"

AMPLIFY_NEURON = best_neuron_by_concept[AMPLIFY_CONCEPT]
#AMPLIFY_NEURON = 94
SUPPRESS_NEURON = best_neuron_by_concept[SUPPRESS_CONCEPT]
#SUPPRESS_NEURON = 5

AMPLIFY_ADD = 3
SUPPRESS_SCALE = 0.5

K = 10

print(
    f"Amplify {AMPLIFY_CONCEPT}: "
    f"Neuron {AMPLIFY_NEURON}"
)

print(
    f"Suppress {SUPPRESS_CONCEPT}: "
    f"Neuron {SUPPRESS_NEURON}"
)

if AMPLIFY_NEURON == SUPPRESS_NEURON:
    raise ValueError(
        f"{AMPLIFY_CONCEPT} and {SUPPRESS_CONCEPT} map to the same "
        f"SAE neuron ({AMPLIFY_NEURON}). They cannot be independently "
        "amplified and suppressed with this one-neuron-per-concept setup."
    )


Amplify Drama: Neuron 16
Suppress Comedy: Neuron 65


In [103]:
CONDITIONS = [
    {
        "Condition": "Baseline",
        "AmplifyAdd": 0.0,
        "SuppressScale": 1.0,
    },
    {
        "Condition": f"{AMPLIFY_CONCEPT} amplify only",
        "AmplifyAdd": AMPLIFY_ADD,
        "SuppressScale": 1.0,
    },
    {
        "Condition": f"{SUPPRESS_CONCEPT} suppress only",
        "AmplifyAdd": 0.0,
        "SuppressScale": SUPPRESS_SCALE,
    },
    {
        "Condition": (
            f"{AMPLIFY_CONCEPT} amplify + "
            f"{SUPPRESS_CONCEPT} suppress"
        ),
        "AmplifyAdd": AMPLIFY_ADD,
        "SuppressScale": SUPPRESS_SCALE,
    },
]

display(
    pd.DataFrame(CONDITIONS)
)


,Condition,AmplifyAdd,SuppressScale
0,Baseline,0.0,1.0
1,Drama amplify only,3.0,1.0
2,Comedy suppress only,0.0,0.5
3,Drama amplify + Comedy suppress,3.0,0.5


## 6. Evaluate both concept changes and NDCG


In [104]:
@torch.no_grad()
def evaluate_two_concept_steering(
    model,
    test_data,
    item_concepts,
    concept_names,
    amplify_neuron,
    suppress_neuron,
    conditions,
    n_items,
    device,
    reconstructed_item_embeddings,
    k=10,
    concept_indices=None,
):
    model.eval()

    if concept_indices is None:
        concept_indices = list(
            range(len(concept_names))
        )

    selected_names = [
        concept_names[i]
        for i in concept_indices
    ]

    concept_tensor = torch.as_tensor(
        item_concepts[:, concept_indices],
        dtype=torch.float32,
        device=device,
    )

    results = []

    for condition in conditions:

        condition_name = condition[
            "Condition"
        ]

        amplify_add = float(
            condition["AmplifyAdd"]
        )

        suppress_scale = float(
            condition["SuppressScale"]
        )

        print("\n" + "=" * 80)
        print(condition_name)
        print(
            f"{AMPLIFY_CONCEPT}: "
            f"Neuron {amplify_neuron}, "
            f"add={amplify_add}"
        )
        print(
            f"{SUPPRESS_CONCEPT}: "
            f"Neuron {suppress_neuron}, "
            f"scale={suppress_scale}"
        )
        print("=" * 80)

        concept_fraction_sum = torch.zeros(
            len(concept_indices),
            dtype=torch.float64,
            device=device,
        )

        concept_hit_sum = torch.zeros(
            len(concept_indices),
            dtype=torch.float64,
            device=device,
        )

        ndcg_sum = 0.0
        ndcg_user_count = 0
        total_users = 0

        for batch in tqdm(
            test_data,
            desc=condition_name,
            leave=False,
        ):

            (
                interaction,
                history_index,
                positive_u,
                positive_i,
            ) = batch

            interaction = interaction.to(
                device
            )

            scores = (
                two_concept_full_sort_predict(
                    model=model,
                    interaction=interaction,
                    amplify_neuron=(
                        amplify_neuron
                    ),
                    amplify_add=(
                        amplify_add
                    ),
                    suppress_neuron=(
                        suppress_neuron
                    ),
                    suppress_scale=(
                        suppress_scale
                    ),
                    reconstructed_item_embeddings=(
                        reconstructed_item_embeddings
                    ),
                )
                .view(-1, n_items)
            )

            # Remove padding item.
            scores[:, 0] = -torch.inf

            # Remove items already consumed by each user.
            if history_index is not None:

                history_index_device = tuple(
                    x.to(device)
                    if torch.is_tensor(x)
                    else x
                    for x in history_index
                )

                scores[
                    history_index_device
                ] = -torch.inf

            topk = torch.topk(
                scores,
                k=k,
                dim=1,
            ).indices

            # -----------------------------
            # Concept metrics
            # -----------------------------

            labels = concept_tensor[
                topk
            ]

            user_fractions = labels.mean(
                dim=1
            )

            user_hits = (
                labels
                .sum(dim=1)
                .gt(0)
                .float()
            )

            concept_fraction_sum += (
                user_fractions
                .sum(dim=0)
                .double()
            )

            concept_hit_sum += (
                user_hits
                .sum(dim=0)
                .double()
            )

            total_users += topk.shape[0]

            # -----------------------------
            # NDCG
            # -----------------------------

            batch_ndcg, valid_mask = (
                compute_ndcg_from_topk(
                    topk=topk,
                    positive_u=positive_u,
                    positive_i=positive_i,
                    k=k,
                )
            )

            ndcg_sum += (
                batch_ndcg[
                    valid_mask
                ]
                .sum()
                .item()
            )

            ndcg_user_count += (
                valid_mask
                .sum()
                .item()
            )

        mean_fractions = (
            concept_fraction_sum
            / total_users
        ).cpu().numpy()

        mean_hits = (
            concept_hit_sum
            / total_users
        ).cpu().numpy()

        mean_ndcg = (
            ndcg_sum
            / ndcg_user_count
        )

        row = {
            "Condition":
                condition_name,

            "AmplifyAdd":
                amplify_add,

            "SuppressScale":
                suppress_scale,

            f"NDCG@{k}":
                float(mean_ndcg),
        }

        for (
            concept_name,
            fraction,
            hit,
        ) in zip(
            selected_names,
            mean_fractions,
            mean_hits,
        ):

            row[
                f"{concept_name}_Fraction@{k}"
            ] = float(fraction)

            row[
                f"{concept_name}_Hit@{k}"
            ] = float(hit)

        results.append(row)

    results_df = pd.DataFrame(
        results
    )

    baseline = (
        results_df[
            results_df["Condition"]
            == "Baseline"
        ]
        .iloc[0]
    )

    baseline_ndcg = float(
        baseline[f"NDCG@{k}"]
    )

    results_df[
        f"Delta_NDCG@{k}"
    ] = (
        results_df[f"NDCG@{k}"]
        - baseline_ndcg
    )

    results_df[
        f"Relative_NDCG_Change@{k}"
    ] = (
        results_df[
            f"Delta_NDCG@{k}"
        ]
        / (baseline_ndcg + 1e-12)
    )

    results_df[
        f"NDCG_Retention@{k}"
    ] = (
        results_df[f"NDCG@{k}"]
        / (baseline_ndcg + 1e-12)
    )

    # Delta of every concept fraction from baseline.
    delta_df = results_df[
        [
            "Condition",
            "AmplifyAdd",
            "SuppressScale",
            f"NDCG@{k}",
            f"Delta_NDCG@{k}",
            f"Relative_NDCG_Change@{k}",
            f"NDCG_Retention@{k}",
        ]
        + [
            f"{name}_Fraction@{k}"
            for name in selected_names
        ]
    ].copy()

    for name in selected_names:

        column = (
            f"{name}_Fraction@{k}"
        )

        delta_df[column] = (
            delta_df[column]
            - float(baseline[column])
        )

    return results_df, delta_df


## 7. Run the experiment

The first 18 columns are evaluated as genre concepts, matching the original notebook.


In [105]:
two_concept_results, two_concept_delta = (
    evaluate_two_concept_steering(
        model=SAE_model,
        test_data=test_data,
        item_concepts=item_concepts_np,
        concept_names=concept_names,
        amplify_neuron=AMPLIFY_NEURON,
        suppress_neuron=SUPPRESS_NEURON,
        conditions=CONDITIONS,
        n_items=test_data.dataset.item_num,
        device=device,
        reconstructed_item_embeddings=(
            reconstructed_item_embeddings
        ),
        k=K,
        concept_indices=list(range(18)),
    )
)



Baseline
Drama: Neuron 16, add=0.0
Comedy: Neuron 65, scale=1.0


Baseline:   0%|          | 0/2 [00:00<?, ?it/s]


Drama amplify only
Drama: Neuron 16, add=3.0
Comedy: Neuron 65, scale=1.0


Drama amplify only:   0%|          | 0/2 [00:00<?, ?it/s]


Comedy suppress only
Drama: Neuron 16, add=0.0
Comedy: Neuron 65, scale=0.5


Comedy suppress only:   0%|          | 0/2 [00:00<?, ?it/s]


Drama amplify + Comedy suppress
Drama: Neuron 16, add=3.0
Comedy: Neuron 65, scale=0.5


Drama amplify + Comedy suppress:   0%|          | 0/2 [00:00<?, ?it/s]

## 8. Main two-concept summary


In [106]:
summary = two_concept_results[
    [
        "Condition",

        f"{AMPLIFY_CONCEPT}_Fraction@{K}",
        f"{AMPLIFY_CONCEPT}_Hit@{K}",

        f"{SUPPRESS_CONCEPT}_Fraction@{K}",
        f"{SUPPRESS_CONCEPT}_Hit@{K}",

        f"NDCG@{K}",
        f"Delta_NDCG@{K}",
        f"Relative_NDCG_Change@{K}",
        f"NDCG_Retention@{K}",
    ]
].copy()

summary[
    "NDCG_Retained_%"
] = (
    summary[
        f"NDCG_Retention@{K}"
    ]
    * 100
)

display(
    summary.round(4)
)


,Condition,Drama_Fraction@10,Drama_Hit@10,Comedy_Fraction@10,Comedy_Hit@10,NDCG@10,Delta_NDCG@10,Relative_NDCG_Change@10,NDCG_Retention@10,NDCG_Retained_%
0,Baseline,0.0963,0.3682,0.1724,0.5467,0.1427,0.0000,0.0000,1.0000,100.0000
1,Drama amplify only,0.1014,0.3740,0.1735,0.5497,0.1398,-0.0030,-0.0209,0.9791,97.9070
2,Comedy suppress only,0.1009,0.3796,0.1598,0.5293,0.1427,-0.0000,-0.0002,0.9998,99.9817
3,Drama amplify + Comedy suppress,0.1067,0.3854,0.1604,0.5321,0.1395,-0.0032,-0.0227,0.9773,97.7332


## 9. Changes in all genres relative to baseline


In [107]:
genre_names = concept_names[:18]

genre_delta_columns = [
    f"{genre}_Fraction@{K}"
    for genre in genre_names
]

display(
    two_concept_delta[
        [
            "Condition",
            f"NDCG@{K}",
            #f"Delta_NDCG@{K}",
            #f"NDCG_Retention@{K}",
        ]
        + genre_delta_columns
    ].round(4)
)


,Condition,NDCG@10,Action_Fraction@10,Adventure_Fraction@10,Animation_Fraction@10,Children's_Fraction@10,Comedy_Fraction@10,Crime_Fraction@10,Documentary_Fraction@10,Drama_Fraction@10,Fantasy_Fraction@10,Film-Noir_Fraction@10,Horror_Fraction@10,Musical_Fraction@10,Mystery_Fraction@10,Romance_Fraction@10,Sci-Fi_Fraction@10,Thriller_Fraction@10,War_Fraction@10,Western_Fraction@10
0,Baseline,0.1427,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,Drama amplify only,0.1398,-0.0006,-0.0000,-0.0007,-0.0014,0.0010,-0.0016,0.0008,0.0051,0.0002,0.0010,0.0006,-0.0007,0.0015,-0.0016,-0.0020,0.0005,-0.0016,-0.0005
2,Comedy suppress only,0.1427,0.0011,0.0001,0.0009,0.0002,-0.0126,0.0012,0.0000,0.0046,0.0014,-0.0001,0.0004,-0.0005,-0.0003,-0.0017,0.0019,0.0025,0.0010,-0.0001
3,Drama amplify + Comedy suppress,0.1395,0.0004,0.0000,0.0002,-0.0011,-0.0120,-0.0003,0.0008,0.0103,0.0016,0.0010,0.0009,-0.0013,0.0012,-0.0034,-0.0002,0.0030,-0.0007,-0.0006


In [108]:
x_all=[]
for c in concept_names[0:18]:
    x=two_concept_delta[two_concept_delta['Condition']=="Drama amplify + Comedy suppress"][f'{c}_Fraction@10'].iloc[0]
    x_all.append((c, x))
x_all

[('Action', 0.00044701873071935697),
 ('Adventure', 3.3113024882133225e-05),
 ('Animation', 0.00016556670334165469),
 ("Children's", -0.0011423818322996437),
 ('Comedy', -0.012003311258278138),
 ('Crime', -0.0003145773679215369),
 ('Documentary', 0.0008112588465608497),
 ('Drama', 0.01033112481729874),
 ('Fantasy', 0.0016059607070013377),
 ('Film-Noir', 0.0010264901925396454),
 ('Horror', 0.0009271482758174687),
 ('Musical', -0.0012582791562111943),
 ('Mystery', 0.0012251658155428627),
 ('Romance', -0.0034271467600437144),
 ('Sci-Fi', -0.00018211011065552563),
 ('Thriller', 0.003029806882340391),
 ('War', -0.0006953662594422552),
 ('Western', -0.0005794714618202884)]

In [109]:
two_concept_delta[two_concept_delta['Condition']=="Drama amplify + Comedy suppress"]['NDCG@10'].iloc[0]

0.13951130039644558

## 10. Inspect only the simultaneous intervention


In [57]:
joint_name = (
    f"{AMPLIFY_CONCEPT} amplify + "
    f"{SUPPRESS_CONCEPT} suppress"
)

joint_row = (
    two_concept_delta[
        two_concept_delta[
            "Condition"
        ] == joint_name
    ]
    .iloc[0]
)

concept_changes = {
    genre:
        joint_row[
            f"{genre}_Fraction@{K}"
        ]
    for genre in genre_names
}

concept_changes = (
    pd.Series(
        concept_changes,
        name=f"Delta Fraction@{K}"
    )
    .sort_values(
        ascending=False
    )
)

print(
    f"Amplified: {AMPLIFY_CONCEPT} "
    f"(Neuron {AMPLIFY_NEURON}, +{AMPLIFY_ADD})"
)

print(
    f"Suppressed: {SUPPRESS_CONCEPT} "
    f"(Neuron {SUPPRESS_NEURON}, "
    f"x{SUPPRESS_SCALE})"
)

print(
    f"NDCG@{K}: "
    f"{joint_row[f'NDCG@{K}']:.4f}"
)

print(
    f"Delta NDCG@{K}: "
    f"{joint_row[f'Delta_NDCG@{K}']:.4f}"
)

print(
    f"NDCG retained: "
    f"{100 * joint_row[f'NDCG_Retention@{K}']:.2f}%"
)

display(
    concept_changes.to_frame()
)


Amplified: Thriller (Neuron 94, +6)
Suppressed: Horror (Neuron 5, x1)
NDCG@10: 0.0945
Delta NDCG@10: -0.0474
NDCG retained: 66.58%


,Delta Fraction@10
Thriller,0.050828
Horror,0.026275
War,0.016076
Western,0.015646
Crime,0.013725
Action,0.010579
Children's,0.010281
Sci-Fi,0.009652
Animation,0.008526
Documentary,0.001010
